In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds

(ds_train, ds_test), ds_info = tfds.load(
    "cifar10",
    split=['train','test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True
)

c:\Users\Shadow\anaconda3\envs\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dl Completed...: 0 url [00:00, ? url/s]
Dl Completed...: 100%|██████████| 1/1 [02:01<00:00, 120.21s/ url]

Dl Completed...: 100%|██████████| 1/1 [02:01<00:00, 121.97s/ url]


Dataset cifar10 downloaded and prepared to C:\Users\Shadow\tensorflow_datasets\cifar10\3.0.2. Subsequent calls will reuse this data.


In [ ]:
def normalize_img(image, label):
    return tf.cast(image, tf.float32) / 255.0, label


AUTOTUNE = tf.data.experimental.AUTOTUNE
BATCH_SIZE =32


def augment(image, label):
    new_height = new_width = 32
    
    # Ensure image has correct dimensions
    image = tf.ensure_shape(image, [None, None, None])  # Ensure 3D
    
    # Resize to target size
    image = tf.image.resize(image, (new_height, new_width))

    if tf.random.uniform((), minval=0, maxval=1) < 0.1:
        grayscale_image = tf.image.rgb_to_grayscale(image)
        image = tf.tile(grayscale_image, [1, 1, 3])

    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.1, upper=0.2)
    image = tf.image.flip_left_right(image)

    return image, label







ds_train = ds_train.map(normalize_img, num_parallel_calls=AUTOTUNE)
ds_train =ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.map(augment, num_parallel_calls=AUTOTUNE)
ds_train = ds_train.batch(BATCH_SIZE)  # Ensure batching is done after mapping
ds_train = ds_train.prefetch(AUTOTUNE)

ds_test = ds_test.batch(BATCH_SIZE)
ds_test = ds_test.prefetch(AUTOTUNE)



model = keras.Sequential(
    [
        keras.Input((32,32,3)),
        layers.Conv2D(4,3, padding="same", activation='relu'),
        layers.Conv2D(8, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(16, 3, padding='same', activation='relu'),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10),
    ]
)


model.compile(
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    optimizer = keras.optimizers.Adam(3e-4),
    metrics=['accuracy'],
)

model.fit(ds_train, epochs=10, verbose=2)
model.evaluate(ds_test)
